# 🖥️ Deploy the gemini-models frontend to Azure App Service

This deploys the small dashboard/chat frontend (`frontend-app/`) as its own Azure App Service, in its **own resource group** — separate from the shared APIM instance and from `lab-gemini-models`. It never touches the shared APIM's resources.

The app's backend keeps the 3 Azure APIM subscription keys **server-side** (as App Settings) and proxies chat calls to the shared gateway — the browser never sees any key, so no CORS policy is needed for this app.

▶️ Run these cells in order after you already have the 3 Azure APIM subscription keys from `geminimodels.ipynb`'s "3️⃣ Get the deployment outputs" cell (this notebook re-fetches them itself below, no copy-paste needed).

### 0️⃣ Initialize variables

In [2]:
import os, sys, json
sys.path.insert(1, '../../../shared')  # add the shared directory to the Python path
import utils

app_resource_group_name = "rg-gemini-models-app"   # separate from lab-gemini-models and from the shared APIM's RG
app_resource_group_location = "westeurope"

shared_apim_name = "apim-shared-pdcibwky2f5ms"
shared_apim_resource_group_name = "rg-shared-apim-gateway-V2"

# Must match the Azure subscription resource names created by
# gemini-shared-resources.bicep (name: 'gemini-models-${subscription.name}').
gemini_models_subscription_ids = {
    "1": "gemini-models-subscription1",
    "2": "gemini-models-subscription2",
    "3": "gemini-models-subscription3",
}

app_service_plan_sku = "B1"  # free tier; bump to "B1" if you need Always On

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 15:06:05.516341 


### 1️⃣ Verify the Azure CLI and the connected Azure subscription

In [3]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 15:06:17.169883 :5s]
👉🏽 Current user: ycure@coem.co
👉🏽 Tenant ID: 0c67c9dd-067e-4ab7-adc3-eac91ce94463
👉🏽 Subscription ID: efbaff8f-21cc-49db-8141-2caaf996decd


### 2️⃣ Fetch the 3 Azure APIM subscription keys

These are the **Azure APIM** subscription keys (the ones `geminimodels.ipynb` prints, used as `Authorization: Bearer ...` against the gateway) — not the Google Gemini API keys. The Google keys stay exactly where they already are, inside APIM's Named Values; this app's backend never touches them directly, it just calls the gateway the same way the notebook's OpenAI SDK cells do.

In [4]:
subscription_keys = {}

for slot, sub_id in gemini_models_subscription_ids.items():
    url = (
        f"https://management.azure.com/subscriptions/{subscription_id}"
        f"/resourceGroups/{shared_apim_resource_group_name}"
        f"/providers/Microsoft.ApiManagement/service/{shared_apim_name}"
        f"/subscriptions/{sub_id}/listSecrets?api-version=2024-06-01-preview"
    )
    output = utils.run(
        f'az rest --method post --url "{url}"',
        f"Retrieved key for {sub_id}",
        f"Failed to retrieve key for {sub_id} — check it was deployed (see geminimodels.ipynb)"
    )
    if output.success and output.json_data:
        subscription_keys[slot] = output.json_data["primaryKey"]
        utils.print_info(f"Subscription {slot}: ****{subscription_keys[slot][-4:]}")

if len(subscription_keys) < 3:
    utils.print_error("No se obtuvieron las 3 keys — revisa el resource group / nombre de la instancia compartida arriba antes de continuar.")

⚙️ Running: az rest --method post --url "https://management.azure.com/subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/providers/Microsoft.ApiManagement/service/apim-shared-pdcibwky2f5ms/subscriptions/gemini-models-subscription1/listSecrets?api-version=2024-06-01-preview" 
✅ Retrieved key for gemini-models-subscription1 ⌚ 15:07:09.417346 :6s]
👉🏽 Subscription 1: ****afd4
⚙️ Running: az rest --method post --url "https://management.azure.com/subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/providers/Microsoft.ApiManagement/service/apim-shared-pdcibwky2f5ms/subscriptions/gemini-models-subscription2/listSecrets?api-version=2024-06-01-preview" 
✅ Retrieved key for gemini-models-subscription2 ⌚ 15:07:12.351578 :2s]
👉🏽 Subscription 2: ****25c1
⚙️ Running: az rest --method post --url "https://management.azure.com/subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/provid

### 3️⃣ Create the App Service (Bicep)

In [5]:
utils.create_resource_group(app_resource_group_name, app_resource_group_location)

bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "appServicePlanSku": { "value": app_service_plan_sku },
        "subscription1Key": { "value": subscription_keys.get("1", "") },
        "subscription2Key": { "value": subscription_keys.get("2", "") },
        "subscription3Key": { "value": subscription_keys.get("3", "") }
    }
}

with open('frontend-app/app-service.params.json', 'w') as f:
    f.write(json.dumps(bicep_parameters))

deployment_name = "gemini-models-frontend"
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {app_resource_group_name} "
    f"--template-file frontend-app/app-service.bicep --parameters frontend-app/app-service.params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed"
)

web_app_name = None
web_app_url = None
if output.success and output.json_data:
    web_app_name = utils.get_deployment_output(output, 'webAppName', 'Web App name')
    web_app_url = utils.get_deployment_output(output, 'webAppUrl', 'Web App URL')

⚙️ Running: az group show --name rg-gemini-models-app 
👉🏽 Resource group rg-gemini-models-app does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name rg-gemini-models-app --location westeurope --tags source=ai-gateway 
✅ Resource group 'rg-gemini-models-app' created ⌚ 15:07:41.225329 :8s]
⚙️ Running: az deployment group create --name gemini-models-frontend --resource-group rg-gemini-models-app --template-file frontend-app/app-service.bicep --parameters frontend-app/app-service.params.json 
✅ Deployment 'gemini-models-frontend' succeeded ⌚ 15:09:00.427309 :19s]


### 🔁 De aquí en adelante: subir cambios de código con la extensión de VS Code

La celda 3 de arriba ya creó el Web App con sus App Settings (keys, URL del gateway). **No hace falta volver a correr las celdas 0-3** salvo que cambies una key, la URL del gateway, el modelo por defecto, o el SKU del plan.

Para subir cambios de código a partir de ahora (editaste `server.js` o `public/index.html`), tienes dos caminos equivalentes:

- Correr la celda 4 de abajo (`az webapp deploy`), o
- En VS Code, con la extensión de Azure App Service instalada: clic derecho sobre la carpeta `frontend-app` → **Deploy to Web App** → elige el Web App llamado como imprimió la celda 3 (`gemini-models-app-...`) → confirma sobrescribir.

Ambos hacen lo mismo (zip + deploy); usa el que te resulte más cómodo en el momento.

### 4️⃣ Deploy the app code (zip deploy)

Zips `frontend-app/` (server.js, package.json, public/) and pushes it with `az webapp deploy`. `SCM_DO_BUILD_DURING_DEPLOYMENT=true` (already set in the Bicep's app settings) makes App Service run `npm install` on its side after unzipping.

In [13]:
import shutil

zip_base = "frontend-app-build"
zip_path = shutil.make_archive(zip_base, 'zip', root_dir='frontend-app')

output = utils.run(
    f'az webapp deploy --resource-group {app_resource_group_name} --name {web_app_name} '
    f'--src-path "{zip_path}" --type zip',
    "App code deployed", "Failed to deploy app code"
)

if web_app_url:
    utils.print_ok(f"Abre tu frontend en: {web_app_url}")
    utils.print_info("Si acabas de desplegar, puede tardar uno o dos minutos en arrancar (más en el tier F1).")

⚙️ Running: az webapp deploy --resource-group rg-gemini-models-app --name gemini-models-app-6kotufhkgj7sq --src-path "c:\Users\ycure\OneDrive - Controles Empresariales SAS\Escritorio\repo\AI-Gateway\labs\gemini-models\gemini-models-app\frontend-app-build.zip" --type zip 
✅ App code deployed ⌚ 09:13:25.480690 :15s]
✅ Abre tu frontend en: https://gemini-models-app-6kotufhkgj7sq.azurewebsites.net ⌚ 09:13:25.480690 
👉🏽 Si acabas de desplegar, puede tardar uno o dos minutos en arrancar (más en el tier F1).


### 🗑️ Clean up

Este App Service vive en su propio resource group (`rg-gemini-models-app`), separado del resto del lab, así que basta con borrar ese resource group cuando termines:

```
az group delete --name rg-gemini-models-app --yes
```

No afecta ni a `lab-gemini-models` ni a la instancia compartida de APIM.